# Transformer from Scratch — Self Attention

## What is Self-Attention?
Self-attention allows every token in a sequence to look at every other token and decide how much focus to give it. Unlike RNNs which process tokens sequentially, self-attention processes the entire sequence **in parallel**.

For example in `"the cat sat on the mat"`, when processing `"sat"`, self-attention lets it look at all other tokens simultaneously and decide which are most relevant to understanding it in context. This is why Transformers understand context far better than RNNs.

---

## The Math
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

| Component | Role |
|-----------|------|
| $QK^T$ | Dot product — how similar is each query to each key |
| $/ \sqrt{d_k}$ | Scaling — prevents large dot products from killing gradients |
| softmax | Normalizes scores to probabilities across all tokens |
| $\times V$ | Weighted sum of values based on those probabilities |

---

## What This Cell Implements

**1. Vocabulary + Tokenization**
Maps raw words → integer indices. Raises clear errors for out-of-vocab tokens.

**2. Embedding Layer**
Lookup table of shape `(vocab_size, d_model)`. Converts token indices into dense vectors. Initialized small `(\times 0.01)` to keep activations stable. In a trained model these are learned via backprop.

**3. QKV Weight Matrices**
Three learned projections `Wq, Wk, Wv` each of shape `(d_model, d_k)`, Xavier initialized `scale = sqrt(2 / (d_model + d_k))` to prevent vanishing/exploding outputs.

| Matrix | Meaning |
|--------|---------|
| $W_Q$ | What each token is **looking for** |
| $W_K$ | What each token **advertises as** |
| $W_V$ | What each token **actually shares** |

**4. QKV Projection** ← current step

In [1]:

import numpy as np



class ScaledDotProductAttention:
    def __init__(self, d_model, d_k, vocab):
        self.d_model = d_model
        self.d_k = d_k
        self.vocab = {word: idx for idx, word in enumerate(vocab)}
        # Embedding table: one row per word in vocab
        self.embedding = np.random.randn(len(vocab), d_model)
        self._init_weights()



    def _init_weights(self):
        scale = np.sqrt(2.0 / (self.d_model + self.d_k))
        self.Wq = np.random.randn(self.d_model, self.d_k) * scale
        self.Wk = np.random.randn(self.d_model, self.d_k) * scale
        self.Wv = np.random.randn(self.d_model, self.d_k) * scale

    def tokenize(self, sentence):
        """
        Convert sentence string to list of token indices.

        Args:
            sentence: Plain string e.g. "the cat sat"
        Returns:
            list of integer indices
        """
        tokens = sentence.lower().split()
        for t in tokens:
            if t not in self.vocab:
                raise ValueError(f"Unknown token: '{t}'. Add it to vocab.")
        return [self.vocab[t] for t in tokens]

    def embed(self, token_ids):
        """
        Look up embedding vectors for each token.

        Args:
            token_ids: list of ints from tokenize()
        Returns:
            X of shape (seq_len, d_model)
        """
        return self.embedding[token_ids]  # fancy indexing

    def compute_qkv(self, sentence):
        """
        Full pipeline: sentence string → Q, K, V matrices.

        Args:
            sentence: Plain string
        Returns:
            Q, K, V: Each of shape (seq_len, d_k)
        """
        token_ids = self.tokenize(sentence)
        X = self.embed(token_ids)
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv
        return Q, K, V


if __name__ == "__main__":
    np.random.seed(42)

    vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast"]

    attention = ScaledDotProductAttention(d_model=8, d_k=4, vocab=vocab)

    # Now works on ANY sentence built from vocab
    for sentence in ["the cat sat", "the dog ran fast", "cat sat on mat"]:
        Q, K, V = attention.compute_qkv(sentence)
        print(f"'{sentence}'  →  Q shape: {Q.shape}")

'the cat sat'  →  Q shape: (3, 4)
'the dog ran fast'  →  Q shape: (4, 4)
'cat sat on mat'  →  Q shape: (4, 4)


## Attention Scores — $QK^T / \sqrt{d_k}$

Each token has a Query (what it looks for) and a Key (what it advertises).
The attention score between two tokens is the dot product of their Q and K vectors — measuring similarity.

$$\text{score}(i, j) = \frac{Q_i \cdot K_j^T}{\sqrt{d_k}}$$

**Why scale by $\sqrt{d_k}$?**
As $d_k$ grows, dot products grow in magnitude pushing softmax into near-zero gradient regions.
Dividing by $\sqrt{d_k}$ keeps the variance of the scores stable regardless of embedding size.

**Output shape:** `(seq_len, seq_len)` — every token scored against every other token.

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [ ] Softmax
- [ ] Weighted Sum → scores $\times$ V
- [ ] Multi-Head Attention
- [ ] Full Encoder Block
- [ ] Full Transformer

In [2]:
# ── Attention Scores ──────────────────────────────────────────────

def compute_attention_scores(Q, K, d_k):
    """
    Compute scaled dot-product attention scores.

    Args:
        Q      : Query matrix of shape (seq_len, d_k)
        K      : Key   matrix of shape (seq_len, d_k)
        d_k    : Key dimension — used for scaling
    Returns:
        scores : shape (seq_len, seq_len)
                 scores[i][j] = how much token i attends to token j
    """
    scores = Q @ K.T                  # (seq_len, seq_len)
    scores = scores / np.sqrt(d_k)    # scale to stabilize gradients
    return scores


# ── Test ──────────────────────────────────────────────────────────

sentence = "the cat sat"
Q, K, V = attention.compute_qkv(sentence)
scores = compute_attention_scores(Q, K, d_k=4)

print(f"Scores shape : {scores.shape}")
print(f"\nRaw attention scores:\n{np.round(scores, 3)}")

Scores shape : (3, 3)

Raw attention scores:
[[ 0.41   1.261 -0.389]
 [ 0.635 -0.905 -0.399]
 [-0.872 -1.856  0.959]]


## Interpreting Attention Scores

The output is a `(seq_len, seq_len)` matrix — read it **row by row**.
Each row = one token asking *"how much should I attend to every other token?"*

$$\text{scores}[i][j] = \text{how much token } i \text{ attends to token } j$$

For `"the cat sat"`:

|  | **"the"** | **"cat"** | **"sat"** |
|---|---|---|---|
| **"the"** | 0.41 | 1.261 | -0.389 |
| **"cat"** | 0.635 | -0.905 | -0.399 |
| **"sat"** | -0.872 | -1.856 | 0.959 |

**Row 0 — "the" looking at everyone**
- `"the" → "cat"` = 1.261 → highest score, attends most to "cat"
- `"the" → "the"` = 0.41 → moderate self-attention
- `"the" → "sat"` = -0.389 → negative, barely attends to "sat"

**Row 1 — "cat" looking at everyone**
- `"cat" → "the"` = 0.635 → attends most to "the" *(makes sense — "the" modifies "cat")*
- `"cat" → "sat"` = -0.399 → weak
- `"cat" → "cat"` = -0.905 → barely attends to itself

**Row 2 — "sat" looking at everyone**
- `"sat" → "sat"` = 0.959 → attends mostly to itself
- `"sat" → "the"` = -0.872 → weak
- `"sat" → "cat"` = -1.856 → lowest score in the whole matrix

---

> **Note:** These are raw scores — not probabilities. They don't sum to 1 and can be negative.
> Softmax (next step) converts each row into a proper probability distribution so they can be used as weights on $V$.

---



## Softmax — Converting Scores to Probabilities

Raw attention scores can be any value — negative, large, small.
Softmax converts each row into a **probability distribution** that sums to 1,
so they can be used as weights on $V$.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Applied **row-wise** — each token's scores across all other tokens sum to 1.

| Property | Meaning |
|----------|---------|
| Output range | $(0, 1)$ — never exactly 0 or 1 |
| Row sum | Always = 1 |
| Large positive score | → close to 1, strong attention |
| Large negative score | → close to 0, weak attention |

**Why not just normalize by dividing by the sum?**
Softmax uses $e^x$ which amplifies differences — a score of 2.0 gets
much more weight than 1.0, making attention **sharp and selective** rather than flat.

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [ ] Weighted Sum → scores $\times$ V
- [ ] Multi-Head Attention
- [ ] Full Encoder Block
- [ ] Full Transformer

In [3]:
# ── Softmax ───────────────────────────────────────────────────────

def softmax(x):
    """
    Row-wise softmax — applied independently to each row.

    Args:
        x      : matrix of shape (seq_len, seq_len)
    Returns:
        probs  : same shape, each row sums to 1
    """
    # subtract row max for numerical stability — prevents e^large from overflowing
    x = x - x.max(axis=1, keepdims=True)
    e_x = np.exp(x)
    return e_x / e_x.sum(axis=1, keepdims=True)


# ── Test ──────────────────────────────────────────────────────────

attn_weights = softmax(scores)

print(f"Attention weights shape : {attn_weights.shape}")
print(f"\nAttention weights (after softmax):\n{np.round(attn_weights, 3)}")
print(f"\nRow sums (should all be 1.0): {np.round(attn_weights.sum(axis=1), 3)}")

Attention weights shape : (3, 3)

Attention weights (after softmax):
[[0.264 0.618 0.119]
 [0.637 0.137 0.226]
 [0.131 0.049 0.819]]

Row sums (should all be 1.0): [1. 1. 1.]


## Weighted Sum — scores $\times$ V

After softmax we have attention weights — a probability distribution per token.
Now we use these weights to compute a **weighted sum of Value vectors**.

$$\text{output} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Each token's output is a blend of **all token Values**, weighted by how much it attends to them.

| | Meaning |
|---|---|
| High attention weight on token $j$ | Token $j$'s Value contributes more to output |
| Low attention weight on token $j$ | Token $j$'s Value contributes less to output |
| Output shape | `(seq_len, d_k)` — same as input, richer in context |

**Example for "cat":**
If "cat" attends 70% to "the", 20% to itself, 10% to "sat" — its output vector =
$$0.7 \times V_{\text{the}} + 0.2 \times V_{\text{cat}} + 0.1 \times V_{\text{sat}}$$

This is how context flows between tokens — "cat" literally absorbs information from "the" and "sat" proportional to attention weights.

---

## Full Formula — all steps together

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

| Step | Operation | Shape |
|------|-----------|-------|
| Input | $X$ | `(seq_len, d_model)` |
| Project | $Q, K, V = XW_Q,\ XW_K,\ XW_V$ | `(seq_len, d_k)` |
| Scores | $QK^T / \sqrt{d_k}$ | `(seq_len, seq_len)` |
| Weights | $\text{softmax}(\text{scores})$ | `(seq_len, seq_len)` |
| Output | $\text{weights} \times V$ | `(seq_len, d_k)` |

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [ ] Multi-Head Attention
- [ ] Full Encoder Block
- [ ] Full Transformer

In [4]:
# ── Weighted Sum ──────────────────────────────────────────────────

def weighted_sum(attn_weights, V):
    """
    Compute context-aware output as weighted sum of Value vectors.

    Args:
        attn_weights : softmax output of shape (seq_len, seq_len)
        V            : Value matrix of shape (seq_len, d_k)
    Returns:
        output       : shape (seq_len, d_k)
                       each token is now a blend of all token values
    """
    return attn_weights @ V


# ── Test ──────────────────────────────────────────────────────────

output = weighted_sum(attn_weights, V)

print(f"Output shape : {output.shape}")
print(f"\nContext-aware output:\n{np.round(output, 3)}")
print(f"\nAttention weights used:\n{np.round(attn_weights, 3)}")

Output shape : (3, 4)

Context-aware output:
[[-0.395 -0.069 -0.175  0.095]
 [-0.241  0.405 -0.479 -0.378]
 [-0.844 -0.075  0.74  -0.611]]

Attention weights used:
[[0.264 0.618 0.119]
 [0.637 0.137 0.226]
 [0.131 0.049 0.819]]


## Multi-Head Attention

### Why Single-Head Attention is Not Enough

Single-head attention computes **one set** of QKV projections — meaning the model
can only look at the sequence through **one lens** at a time.

Consider `"The animal didn't cross the street because it was too tired"`.
To resolve what `"it"` refers to, the model needs to track **multiple relationships simultaneously**:

| Head | What it could learn to focus on |
|------|----------------------------------|
| Head 1 | Syntactic — `"it"` → `"animal"` (subject agreement) |
| Head 2 | Semantic — `"tired"` → `"animal"` (animals get tired, not streets) |
| Head 3 | Positional — nearby tokens |
| Head 4 | Coreference — pronoun → noun resolution |

With a single head, the model is forced to **average all these signals into one**,
losing the fine-grained structure. Multi-head attention runs $h$ attention heads
in parallel, each learning a different aspect of relationships.

---

### How it Works

Instead of one big projection of size `d_model → d_k`, we split into $h$ smaller heads,
each of size `d_model → d_model/h`.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \cdot W^O$$

$$\text{where head}_i = \text{Attention}(QW_i^Q,\ KW_i^K,\ VW_i^V)$$

| Symbol | Meaning |
|--------|---------|
| $h$ | Number of heads |
| $d_k = d_{model} / h$ | Dimension per head |
| $W_i^Q, W_i^K, W_i^V$ | Separate learned projections per head |
| $W^O$ | Output projection to merge all heads back to `d_model` |

**Shape flow:**

| Step | Shape |
|------|-------|
| Input $X$ | `(seq_len, d_model)` |
| Each head output | `(seq_len, d_k)` |
| After concat all heads | `(seq_len, h × d_k)` = `(seq_len, d_model)` |
| After output projection $W^O$ | `(seq_len, d_model)` |

---

### Key Insight
Each head gets its **own** $W^Q, W^K, W^V$ — so each head learns to attend
to completely different patterns. The output projection $W^O$ then learns
how to best combine all those perspectives into one rich representation.

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [ ] Full Encoder Block
- [ ] Full Transformer

In [5]:
# ── Multi-Head Attention ──────────────────────────────────────────

class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model   : total embedding dimension (e.g. 8)
            num_heads : number of parallel attention heads (e.g. 2)
        """
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads   # dimension per head

        # each head gets its own Wq, Wk, Wv
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.Wq = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wk = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wv = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]

        # output projection — merges all heads back to d_model
        self.Wo = np.random.randn(d_model, d_model) * np.sqrt(2.0 / (d_model + d_model))

    def _single_head_attention(self, X, Wq, Wk, Wv):
        """
        Full single-head attention — everything we built so far.

        Args:
            X        : (seq_len, d_model)
            Wq,Wk,Wv : (d_model, d_k)
        Returns:
            output   : (seq_len, d_k)
            weights  : (seq_len, seq_len) — for inspection
        """
        Q = X @ Wq
        K = X @ Wk
        V = X @ Wv

        scores = Q @ K.T / np.sqrt(self.d_k)

        # numerical stability trick
        scores = scores - scores.max(axis=1, keepdims=True)
        weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)

        output = weights @ V
        return output, weights

    def forward(self, X):
        """
        Run all heads in parallel, concat, then project.

        Args:
            X      : (seq_len, d_model)
        Returns:
            output : (seq_len, d_model)
            heads  : list of attention weight matrices — one per head
        """
        head_outputs = []
        head_weights = []

        for i in range(self.num_heads):
            out, weights = self._single_head_attention(X, self.Wq[i], self.Wk[i], self.Wv[i])
            head_outputs.append(out)      # each: (seq_len, d_k)
            head_weights.append(weights)  # each: (seq_len, seq_len)

        # concat all heads along last axis → (seq_len, d_model)
        concat = np.concatenate(head_outputs, axis=-1)

        # final linear projection
        output = concat @ self.Wo         # (seq_len, d_model)

        return output, head_weights


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

vocab    = ["the", "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"]
sentence = "the animal didn't cross the street because it was too tired"

# simple embedding for test
d_model   = 8
num_heads = 2   # d_k = 8/2 = 4 per head

embedding_table = np.random.randn(len(vocab), d_model)
token_ids       = [vocab.index(w) for w in sentence.split()]
X               = embedding_table[token_ids]   # (11, 8)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
output, head_weights = mha.forward(X)

print(f"Input  shape : {X.shape}")
print(f"Output shape : {output.shape}")
print(f"\nHead 1 attention weights:\n{np.round(head_weights[0], 3)}")
print(f"\nHead 2 attention weights:\n{np.round(head_weights[1], 3)}")
print(f"\nNotice how Head 1 and Head 2 attend to different tokens — each learned a different pattern.")

Input  shape : (11, 8)
Output shape : (11, 8)

Head 1 attention weights:
[[0.041 0.141 0.216 0.042 0.041 0.255 0.053 0.065 0.052 0.058 0.035]
 [0.129 0.02  0.01  0.149 0.129 0.006 0.084 0.053 0.11  0.094 0.218]
 [0.054 0.066 0.086 0.213 0.054 0.03  0.071 0.067 0.149 0.083 0.128]
 [0.269 0.018 0.005 0.026 0.269 0.016 0.095 0.06  0.038 0.078 0.126]
 [0.041 0.141 0.216 0.042 0.041 0.255 0.053 0.065 0.052 0.058 0.035]
 [0.123 0.007 0.002 0.193 0.123 0.001 0.066 0.029 0.101 0.107 0.248]
 [0.126 0.05  0.034 0.112 0.126 0.033 0.101 0.083 0.098 0.101 0.134]
 [0.123 0.069 0.047 0.086 0.123 0.06  0.097 0.079 0.087 0.122 0.106]
 [0.128 0.079 0.062 0.058 0.128 0.101 0.102 0.099 0.069 0.086 0.088]
 [0.084 0.111 0.12  0.055 0.084 0.161 0.082 0.09  0.069 0.077 0.067]
 [0.01  0.13  0.439 0.016 0.01  0.285 0.018 0.032 0.028 0.015 0.015]]

Head 2 attention weights:
[[0.011 0.067 0.226 0.02  0.011 0.198 0.072 0.214 0.153 0.003 0.025]
 [0.064 0.132 0.161 0.094 0.064 0.048 0.028 0.019 0.067 0.124 0.198]
 [

## Cell 6 — Positional Encoding

### Why Do We Need It?

Self-attention has no sense of order — it treats the input as a **set, not a sequence**.
For the model, `"cat sat on mat"` and `"mat on sat cat"` look identical without position info.

Positional Encoding injects **order information** directly into the embeddings
before they enter the encoder.

---

### The Formula

From the original paper, positional encoding uses sine and cosine functions:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

| Symbol | Meaning |
|--------|---------|
| $pos$ | Position of the token in the sequence (0, 1, 2, ...) |
| $i$ | Dimension index (0, 1, 2, ... $d_{model}/2$) |
| $d_{model}$ | Embedding dimension |

**Even dimensions** → sine, **odd dimensions** → cosine.

---

### Why Sine and Cosine?

| Property | Benefit |
|----------|---------|
| Values always in $[-1, 1]$ | Won't distort embedding magnitudes |
| Each position has a unique pattern | Model can distinguish any two positions |
| Smooth and continuous | Nearby positions have similar encodings |
| Generalizes to unseen lengths | Works for sequences longer than seen in training |

---

### How it is Applied

$$X_{input} = \text{Embedding}(x) + PE$$

The positional encoding is simply **added** to the token embedding —
same shape `(seq_len, d_model)`, no extra parameters needed.

> In the original Transformer, 6 encoder blocks are stacked — each receives
> the output of the previous block. Positional encoding is only added **once**
> at the very beginning before block 1. For our implementation we use **1 block**.

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [x] Positional Encoding
- [ ] Input Embedding + Positional Encoding
- [ ] Add & Norm 1
- [ ] Feed Forward
- [ ] Add & Norm 2
- [ ] Full Encoder

In [6]:
# ── Positional Encoding ───────────────────────────────────────────

class PositionalEncoding:
    def __init__(self, d_model, max_seq_len=100):
        """
        Args:
            d_model     : embedding dimension
            max_seq_len : maximum sequence length to precompute for
        """
        self.d_model = d_model

        # precompute PE table of shape (max_seq_len, d_model)
        PE = np.zeros((max_seq_len, d_model))

        positions = np.arange(max_seq_len).reshape(-1, 1)        # (max_seq_len, 1)
        dims      = np.arange(0, d_model, 2)                     # even indices only

        div_term  = np.power(10000.0, dims / d_model)            # (d_model/2,)

        PE[:, 0::2] = np.sin(positions / div_term)               # even dims → sin
        PE[:, 1::2] = np.cos(positions / div_term)               # odd  dims → cos

        self.PE = PE

    def forward(self, X):
        """
        Add positional encoding to input embeddings.

        Args:
            X      : (seq_len, d_model)
        Returns:
            output : (seq_len, d_model) — embeddings + position info
        """
        seq_len = X.shape[0]
        return X + self.PE[:seq_len, :]   # slice PE to actual seq length


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

d_model = 8
pe      = PositionalEncoding(d_model=d_model, max_seq_len=100)

# dummy embeddings for 3 tokens
X_test  = np.random.randn(3, d_model)
X_out   = pe.forward(X_test)

print(f"Input  shape : {X_test.shape}")
print(f"Output shape : {X_out.shape}")
print(f"\nPositional Encoding table (first 5 positions):\n{np.round(pe.PE[:5], 3)}")
print(f"\nBefore PE:\n{np.round(X_test, 3)}")
print(f"\nAfter  PE:\n{np.round(X_out,  3)}")

Input  shape : (3, 8)
Output shape : (3, 8)

Positional Encoding table (first 5 positions):
[[ 0.     1.     0.     1.     0.     1.     0.     1.   ]
 [ 0.841  0.54   0.1    0.995  0.01   1.     0.001  1.   ]
 [ 0.909 -0.416  0.199  0.98   0.02   1.     0.002  1.   ]
 [ 0.141 -0.99   0.296  0.955  0.03   1.     0.003  1.   ]
 [-0.757 -0.654  0.389  0.921  0.04   0.999  0.004  1.   ]]

Before PE:
[[ 0.497 -0.138  0.648  1.523 -0.234 -0.234  1.579  0.767]
 [-0.469  0.543 -0.463 -0.466  0.242 -1.913 -1.725 -0.562]
 [-1.013  0.314 -0.908 -1.412  1.466 -0.226  0.068 -1.425]]

After  PE:
[[ 0.497  0.862  0.648  2.523 -0.234  0.766  1.579  1.767]
 [ 0.372  1.083 -0.364  0.529  0.252 -0.913 -1.724  0.438]
 [-0.104 -0.102 -0.709 -0.432  1.486  0.774  0.07  -0.425]]


## Cell 7 — Input Embedding + Positional Encoding

### The Problem

A Transformer cannot work with raw text — it only understands numbers.
So before anything else we need to convert each word into a vector.
But just converting words to vectors is not enough — we also need to tell
the model **where** each word sits in the sentence.

This cell solves both problems.

---

### Step 1 — Tokenization

Split the sentence into words and map each word to an integer index
using a fixed vocabulary.

```
"the cat sat"  →  ["the", "cat", "sat"]  →  [0, 1, 2]
```

These integers are just IDs — they carry no meaning yet.

---

### Step 2 — Embedding Lookup

We have an **embedding table** of shape `(vocab_size, d_model)` —
one learned vector per word. We use the token IDs to look up the
corresponding row for each token.

```
token 0 ("the") → row 0 of table → [0.5, 1.2, -0.3, ...]
token 1 ("cat") → row 1 of table → [1.1, -0.4, 0.8, ...]
token 2 ("sat") → row 2 of table → [-0.2, 0.9, 0.6, ...]
```

Result shape: `(seq_len, d_model)`.

These vectors are **randomly initialized** and updated during training —
over time similar words end up with similar vectors.

We also scale embeddings by $\sqrt{d_{model}}$ before adding PE.
This is done in the original paper because PE values are always in $[-1, 1]$
and without scaling they can dominate the embedding signal when `d_model` is large.

---

### Step 3 — Add Positional Encoding

Attention has no sense of word order — `"cat sat"` and `"sat cat"` look
identical to it. So we add the positional encoding vector (computed in Cell 6)
directly to each token embedding:

$$X_{input} = \text{Embedding}(x) \times \sqrt{d_{model}} + PE$$

Both have the same shape `(seq_len, d_model)` so addition works elementwise.
Now each token vector carries both **meaning** and **position**.

This `X_input` is what enters the first encoder block.

> In the original Transformer 6 encoder blocks are stacked one after another.
> Positional encoding is added **only once here** — it naturally propagates
> through all blocks via residual connections.
> For our implementation we use **1 block** to keep it simple.

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [x] Positional Encoding
- [x] Input Embedding + Positional Encoding
- [ ] Add & Norm 1
- [ ] Feed Forward
- [ ] Add & Norm 2
- [ ] Full Encoder

In [7]:
# ── Input Embedding + Positional Encoding ────────────────────────

class InputEmbedding:
    def __init__(self, vocab, d_model):
        """
        Args:
            vocab   : list of words e.g. ["the", "cat", "sat"]
            d_model : embedding dimension
        """
        self.vocab   = {word: idx for idx, word in enumerate(vocab)}
        self.d_model = d_model

        # embedding table — one row per word, learned during training
        self.embedding_table = np.random.randn(len(vocab), d_model)

        # scale factor from original paper — prevents PE dominating embeddings
        self.scale = np.sqrt(d_model)

        self.pe = PositionalEncoding(d_model=d_model, max_seq_len=100)

    def tokenize(self, sentence):
        """
        Convert sentence string → list of token indices.

        Args:
            sentence : plain string e.g. "the cat sat"
        Returns:
            list of integer indices
        """
        tokens = sentence.lower().split()
        for t in tokens:
            if t not in self.vocab:
                raise ValueError(f"Unknown token: '{t}'. Add it to vocab.")
        return [self.vocab[t] for t in tokens]

    def forward(self, sentence):
        """
        Full pipeline: raw sentence → embedding + positional encoding.

        Args:
            sentence : plain string
        Returns:
            X        : (seq_len, d_model) — ready for encoder
            tokens   : list of token strings — for inspection
        """
        tokens    = sentence.lower().split()
        token_ids = self.tokenize(sentence)

        # step 1 — lookup and scale embeddings
        X = self.embedding_table[token_ids] * self.scale   # (seq_len, d_model)

        # step 2 — add positional encoding
        X = self.pe.forward(X)                             # (seq_len, d_model)

        return X, tokens


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

vocab    = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast",
            "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"]
d_model  = 8
sentence = "the cat sat on mat"

embed     = InputEmbedding(vocab=vocab, d_model=d_model)
X, tokens = embed.forward(sentence)

print(f"Tokens       : {tokens}")
print(f"Output shape : {X.shape}")
print(f"\nEmbedding table (first 3 words):\n{np.round(embed.embedding_table[:3], 3)}")
print(f"\nAfter scaling + PE:\n{np.round(X, 3)}")
print(f"\nX is now ready to enter the encoder block.")

Tokens       : ['the', 'cat', 'sat', 'on', 'mat']
Output shape : (5, 8)

Embedding table (first 3 words):
[[ 0.497 -0.138  0.648  1.523 -0.234 -0.234  1.579  0.767]
 [-0.469  0.543 -0.463 -0.466  0.242 -1.913 -1.725 -0.562]
 [-1.013  0.314 -0.908 -1.412  1.466 -0.226  0.068 -1.425]]

After scaling + PE:
[[ 1.405  0.609  1.832  5.308 -0.662  0.338  4.467  3.171]
 [-0.486  2.075 -1.211 -0.322  0.694 -4.412 -4.878 -0.59 ]
 [-1.955  0.473 -2.37  -3.015  4.165  0.361  0.193 -3.03 ]
 [-1.399 -0.676 -2.96   2.018 -1.669  0.175 -1.699  6.239]
 [-0.795 -3.645  2.716 -2.532  0.631 -4.544 -3.753  1.557]]

X is now ready to enter the encoder block.


## Cell 8 — Multi-Head Attention + Add & Norm 1

### What Happens Here?

This is the first sublayer of the encoder block.
We pass `X_input` (from Cell 7) through Multi-Head Attention,
then apply a **residual connection** and **Layer Normalization**.

```
## Cell 8 — Multi-Head Attention + Add & Norm 1

### What Happens Here?

This is the first sublayer of the encoder block.
We pass `X_input` (from Cell 7) through Multi-Head Attention,
then apply a **residual connection** and **Layer Normalization**.

```
X_input  ──────────────────────────────┐
   │                                   │  (residual — skip connection)
   ▼                                   │
Multi-Head Attention                   │
   │                                   │
   ▼                                   │
MHA_output + X_input  ←────────────────┘
   │
   ▼
Layer Norm
   │
   ▼
X_norm1   →  goes into Feed Forward (Cell 9)
```

---

### Why Residual Connection?

Without it, as we stack more encoder blocks the gradient signal
weakens as it travels back through many layers — **vanishing gradient problem**.

The residual connection creates a **shortcut** — gradients can flow directly
through the addition without passing through attention weights.
Same idea as ResNets in computer vision.

$$\text{output} = \text{LayerNorm}(X + \text{MHA}(X))$$

The model now has two paths to learn from:
- The **transformed path** — what MHA learned
- The **identity path** — the original input unchanged

---

### Why Layer Norm?

After adding the residual, values can vary wildly in scale across tokens.
Layer Norm brings them back to a stable range by normalizing
**each token independently** across its feature dimension.

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma + \epsilon} \cdot \gamma + \beta$$

| Symbol | Meaning |
|--------|---------|
| $\mu$ | Mean across `d_model` features for one token |
| $\sigma$ | Std across `d_model` features for one token |
| $\gamma, \beta$ | Learned scale and shift — initialized to 1 and 0 |
| $\epsilon$ | Small constant to prevent division by zero |

> Unlike BatchNorm which normalizes across the batch dimension,
> LayerNorm normalizes across the feature dimension —
> making it completely independent of batch size and sequence length.
> This is why it is preferred in NLP.

---

Why Layer Norm and Not Batch Norm?
Batch Norm normalizes across the batch dimension — column wise across all sentences in the batch.
In NLP sentences have variable lengths so we pad shorter sentences with zeros to make them the same length. Now when Batch Norm computes mean and std column wise, those zero padding values are included in the calculation — polluting the statistics with values that carry no real meaning.

Batch (after padding):

"the cat sat"     →  [0.5,  1.2, -0.3,  0.8]
"dog ran"         →  [1.1, -0.4,  0.0,  0.0]  ← padded zeros
"the animal ..."  →  [0.2,  0.9,  0.6, -0.1]

Batch Norm computes mean/std column wise — zeros from padding
skew every single column statistic.

Layer Norm normalizes across the feature dimension — row wise, one token at a time. Each token is normalized using only its own d_model features. Padding tokens are on separate rows so they never interfere with real token statistics.

Layer Norm — each row normalized independently:

"the"  →  [0.5,  1.2, -0.3,  0.8]  ← normalized using only these 4 values
"cat"  →  [1.1, -0.4,  0.3,  0.6]  ← normalized using only these 4 values
[PAD]  →  [0.0,  0.0,  0.0,  0.0]  ← isolated, never touches other rows

This is why every modern NLP architecture — BERT, GPT, T5 — uses Layer Norm and not Batch Norm.

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [x] Positional Encoding
- [x] Input Embedding + Positional Encoding
- [x] Multi-Head Attention + Add & Norm 1
- [ ] Feed Forward
- [ ] Add & Norm 2
- [ ] Full Encoder

In [8]:
# ── Layer Norm ────────────────────────────────────────────────────

class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        """
        Args:
            d_model : feature dimension to normalize across
            eps     : small constant for numerical stability
        """
        self.eps   = eps
        self.gamma = np.ones(d_model)    # learned scale — init 1
        self.beta  = np.zeros(d_model)   # learned shift — init 0

    def forward(self, x):
        """
        Normalize each token independently across feature dimension.

        Args:
            x      : (seq_len, d_model)
        Returns:
            output : (seq_len, d_model)
        """
        mean  = x.mean(axis=-1, keepdims=True)    # per token mean
        std   = x.std(axis=-1, keepdims=True)     # per token std
        x_norm = (x - mean) / (std + self.eps)    # normalize
        return self.gamma * x_norm + self.beta    # scale and shift


# ── Multi-Head Attention + Add & Norm 1 ──────────────────────────

class MHAWithAddNorm:
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model   : embedding dimension
            num_heads : number of parallel attention heads
        """
        self.mha   = MultiHeadAttention(d_model, num_heads)
        self.norm  = LayerNorm(d_model)

    def forward(self, X):
        """
        Args:
            X       : (seq_len, d_model) — output of InputEmbedding
        Returns:
            X_norm1 : (seq_len, d_model) — ready for Feed Forward
        """
        # step 1 — multi head attention
        mha_out, attn_weights = self.mha.forward(X)   # (seq_len, d_model)

        # step 2 — residual connection + layer norm
        X_norm1 = self.norm.forward(X + mha_out)      # (seq_len, d_model)

        return X_norm1, attn_weights


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

vocab    = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast",
            "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"]
d_model   = 8
num_heads = 2
sentence  = "the cat sat on mat"

# get X from Cell 7
embed         = InputEmbedding(vocab=vocab, d_model=d_model)
X, tokens     = embed.forward(sentence)

# pass through MHA + Add & Norm 1
mha_addnorm   = MHAWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm1, attn_weights = mha_addnorm.forward(X)

print(f"Tokens         : {tokens}")
print(f"Input  shape   : {X.shape}")
print(f"Output shape   : {X_norm1.shape}")
print(f"\nAttention weights — Head 1:\n{np.round(attn_weights[0], 3)}")
print(f"\nAttention weights — Head 2:\n{np.round(attn_weights[1], 3)}")
print(f"\nX_norm1 (ready for Feed Forward):\n{np.round(X_norm1, 3)}")

Tokens         : ['the', 'cat', 'sat', 'on', 'mat']
Input  shape   : (5, 8)
Output shape   : (5, 8)

Attention weights — Head 1:
[[0.979 0.    0.    0.    0.021]
 [0.    0.82  0.155 0.024 0.   ]
 [0.    0.009 0.982 0.008 0.   ]
 [0.08  0.549 0.    0.    0.371]
 [0.    0.    0.999 0.    0.   ]]

Attention weights — Head 2:
[[0.833 0.    0.001 0.166 0.   ]
 [0.    0.    0.994 0.005 0.001]
 [0.    0.476 0.069 0.    0.454]
 [0.    0.    0.    1.    0.   ]
 [0.    0.004 0.995 0.    0.001]]

X_norm1 (ready for Feed Forward):
[[-7.720e-01 -1.610e-01  5.500e-02  2.116e+00 -4.310e-01 -1.205e+00
  -5.690e-01  9.670e-01]
 [-1.260e+00  1.659e+00 -9.480e-01  2.510e-01  1.023e+00 -3.540e-01
  -1.039e+00  6.680e-01]
 [ 1.000e-03  4.030e-01 -3.790e-01 -2.203e+00  9.270e-01  9.950e-01
   8.200e-01 -5.640e-01]
 [-7.520e-01 -1.750e-01 -8.700e-02  8.070e-01 -9.800e-01  3.730e-01
  -1.224e+00  2.037e+00]
 [ 4.300e-02 -1.241e+00  1.012e+00 -1.408e+00  9.170e-01 -9.740e-01
   4.380e-01  1.213e+00]]


## Cell 9 — Feed Forward Network

### What Happens Here?

After Add & Norm 1, each token has a context-aware representation —
it knows what to attend to. But attention is a **purely linear operation**.
The Feed Forward Network (FFN) adds **non-linearity** — letting the model
learn complex token-level transformations that attention alone cannot capture.

```
X_norm1  (from Add & Norm 1)
   │
   ▼
Linear Layer 1  →  expand  (d_model → d_ff)
   │
   ▼
ReLU
   │
   ▼
Linear Layer 2  →  compress back  (d_ff → d_model)
   │
   ▼
FFN_output  →  goes into Add & Norm 2 (Cell 10)
```

---

### The Formula

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

| Component | Shape | Purpose |
|-----------|-------|---------|
| $W_1$ | `(d_model, d_ff)` | Expand to higher dimension |
| $W_2$ | `(d_ff, d_model)` | Compress back to d_model |
| ReLU | — | Non-linearity — kills negative values |
| $d_{ff}$ | $4 \times d_{model}$ | Standard from original paper |

---

### Why Expand Then Compress?

Expanding to a higher dimension (`d_ff = 4 × d_model`) gives the model
a **larger representational space** to learn complex transformations in.
Compressing back ensures the output shape stays `(seq_len, d_model)`
so it can flow cleanly into the next block.

Think of it as: *zoom out to see more possibilities, zoom back in to pick the best one.*

---

### Important — FFN is Token-Wise

FFN is applied **independently to each token** using the same weights.
It does not mix information across tokens — that is attention's job.
FFN's job is purely to transform each token's representation individually.

| | Attention | FFN |
|---|---|---|
| Mixes tokens? | Yes | No |
| Non-linear? | No | Yes |
| Purpose | Context across tokens | Transform each token |

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [x] Positional Encoding
- [x] Input Embedding + Positional Encoding
- [x] Multi-Head Attention + Add & Norm 1
- [x] Feed Forward
- [ ] Add & Norm 2
- [ ] Full Encoder

In [9]:
# ── Feed Forward Network ──────────────────────────────────────────

class FeedForward:
    def __init__(self, d_model, d_ff):
        """
        Args:
            d_model : input and output dimension
            d_ff    : inner dimension — typically 4 × d_model
        """
        self.d_model = d_model
        self.d_ff    = d_ff

        # xavier initialization for both layers
        scale1 = np.sqrt(2.0 / (d_model + d_ff))
        scale2 = np.sqrt(2.0 / (d_ff + d_model))

        self.W1 = np.random.randn(d_model, d_ff) * scale1   # expand
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * scale2   # compress
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        """
        Apply FFN independently to each token.

        Args:
            x      : (seq_len, d_model)
        Returns:
            output : (seq_len, d_model)
        """
        # step 1 — expand + ReLU
        x = np.maximum(0, x @ self.W1 + self.b1)   # (seq_len, d_ff)

        # step 2 — compress back
        x = x @ self.W2 + self.b2                  # (seq_len, d_model)

        return x

## Cell 10 — Add & Norm 2

### What Happens Here?

Exact same pattern as Add & Norm 1 — but this time wrapping the **Feed Forward Network**.
Take the FFN output, add the residual from before FFN, then normalize.

```
X_norm1  ──────────────────────────────┐
   │                                   │  (residual — skip connection)
   ▼                                   │
Feed Forward Network                   │
   │                                   │
   ▼                                   │
FFN_output + X_norm1  ←────────────────┘
   │
   ▼
Layer Norm
   │
   ▼
X_norm2   →  final output of one encoder block
```

---

### Why Again?

Each sublayer in the encoder — attention and FFN — gets its own
residual connection and layer norm. This is the **Post-LN** pattern
from the original paper:

$$X_{norm2} = \text{LayerNorm}(X_{norm1} + \text{FFN}(X_{norm1}))$$

| | Add & Norm 1 | Add & Norm 2 |
|---|---|---|
| Wraps | Multi-Head Attention | Feed Forward |
| Residual from | `X_input` | `X_norm1` |
| Output | `X_norm1` | `X_norm2` |

`X_norm2` is the final output of one complete encoder block.
In the original Transformer this feeds into the next encoder block —
repeated 6 times. In our implementation it is the final encoder output.

---

## Roadmap
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [x] Positional Encoding
- [x] Input Embedding + Positional Encoding
- [x] Multi-Head Attention + Add & Norm 1
- [x] Feed Forward
- [x] Add & Norm 2
- [ ] Full Encoder

In [10]:
# ── Add & Norm 2 ──────────────────────────────────────────────────

class FFNWithAddNorm:
    def __init__(self, d_model, d_ff):
        """
        Args:
            d_model : embedding dimension
            d_ff    : feed forward inner dimension
        """
        self.ffn  = FeedForward(d_model, d_ff)
        self.norm = LayerNorm(d_model)

    def forward(self, X_norm1):
        """
        Args:
            X_norm1 : (seq_len, d_model) — output of Add & Norm 1
        Returns:
            X_norm2 : (seq_len, d_model) — final encoder block output
        """
        # step 1 — feed forward
        ffn_out = self.ffn.forward(X_norm1)                       # (seq_len, d_model)

        # step 2 — residual connection + layer norm
        X_norm2 = self.norm.forward(X_norm1 + ffn_out)            # (seq_len, d_model)

        return X_norm2

## Cell 11 — Full Encoder

### What We Built

Looking at the encoder diagram — here is exactly what we implemented cell by cell:

```
Raw Sentence
     │
     ▼
Cell 7 — Input Embedding + Positional Encoding
     │
     ▼
Cell 8 — Multi-Head Attention + Add & Norm 1
     │
     ▼
Cell 9 — Feed Forward
     │
     ▼
Cell 10 — Add & Norm 2
     │
     ▼
Encoder Output  →  (seq_len, d_model)
```

In the original paper this entire block is stacked **6 times (Nx = 6)**.
Each block receives the output of the previous one as its input.
For our implementation we use **1 block** — adding more is just a loop.

---

### Full Shape Flow

| Cell | Operation | Shape |
|------|-----------|-------|
| Cell 7 | Tokenize → Embed → + PE | `(seq_len, d_model)` |
| Cell 8 | MHA → + residual → LayerNorm | `(seq_len, d_model)` |
| Cell 9 | FFN (expand → ReLU → compress) | `(seq_len, d_model)` |
| Cell 10 | + residual → LayerNorm | `(seq_len, d_model)` |

> Shape never changes through the entire encoder — always `(seq_len, d_model)`.
> This is what makes stacking Nx blocks clean and simple.

---

### What the Encoder Output Represents

Each row in the output matrix is a **context-aware representation** of that token.
Unlike the raw embedding which only captures word meaning in isolation,
the encoder output captures meaning **in the context of the full sentence**.

For example in `"the animal didn't cross the street because it was too tired"`,
the vector for `"it"` in the raw embedding carries no information about
what `"it"` refers to. After the encoder, `"it"`'s vector has absorbed
context from `"animal"` and `"tired"` — it now implicitly represents the animal.

---

## Roadmap — Encoder Complete
- [x] Tokenization + Embeddings
- [x] QKV Projection
- [x] Attention Scores → $QK^T / \sqrt{d_k}$
- [x] Softmax
- [x] Weighted Sum → scores $\times$ V
- [x] Multi-Head Attention
- [x] Positional Encoding
- [x] Input Embedding + Positional Encoding
- [x] Multi-Head Attention + Add & Norm 1
- [x] Feed Forward
- [x] Add & Norm 2
- [x] Full Encoder ✓

## Roadmap — Decoder (Next Section)
- [ ] Output Embedding + Positional Encoding
- [ ] Masked Multi-Head Attention + Add & Norm
- [ ] Cross Attention + Add & Norm
- [ ] Feed Forward + Add & Norm
- [ ] Linear + Softmax
- [ ] Full Decoder

In [11]:
# ── Full Encoder ──────────────────────────────────────────────────
import numpy as np
class Encoder:
    def __init__(self, vocab, d_model, num_heads, d_ff, num_blocks=1):
        """
        Args:
            vocab      : list of words
            d_model    : embedding dimension
            num_heads  : number of attention heads
            d_ff       : feed forward inner dimension (typically 4 × d_model)
            num_blocks : number of stacked encoder blocks (6 in original paper)
        """
        self.embedding  = InputEmbedding(vocab=vocab, d_model=d_model)
        self.blocks     = [
            {
                "mha_norm" : MHAWithAddNorm(d_model, num_heads),
                "ffn_norm" : FFNWithAddNorm(d_model, d_ff)
            }
            for _ in range(num_blocks)
        ]

    def forward(self, sentence):
        """
        Full encoder pipeline — raw sentence to context aware output.

        Args:
            sentence : plain string
        Returns:
            X        : (seq_len, d_model) — encoder output
            tokens   : list of token strings
        """
        # cell 7 — input embedding + positional encoding
        X, tokens = self.embedding.forward(sentence)

        # cell 8 → 10 — repeated for each encoder block
        for block in self.blocks:
            X, _ = block["mha_norm"].forward(X)   # MHA + Add & Norm 1
            X    = block["ffn_norm"].forward(X)    # FFN + Add & Norm 2

        return X, tokens


# ── Full End to End Test ──────────────────────────────────────────

np.random.seed(42)

vocab = [
    "the", "cat", "sat", "on", "mat", "dog", "ran", "fast",
    "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"
]

d_model   = 8
num_heads = 2
d_ff      = 32   # 4 × d_model
num_blocks = 1   # use 6 for full transformer

encoder = Encoder(
    vocab       = vocab,
    d_model     = d_model,
    num_heads   = num_heads,
    d_ff        = d_ff,
    num_blocks  = num_blocks
)

test_sentences = [
    "the cat sat on mat",
    "the dog ran fast",
    "the animal didn't cross the street"
]

for sentence in test_sentences:
    output, tokens = encoder.forward(sentence)
    print(f"Input  : '{sentence}'")
    print(f"Tokens : {tokens}")
    print(f"Output shape : {output.shape}")
    print(f"Encoder output:\n{np.round(output, 3)}")
    print("─" * 60)

Input  : 'the cat sat on mat'
Tokens : ['the', 'cat', 'sat', 'on', 'mat']
Output shape : (5, 8)
Encoder output:
[[-1.243 -0.068 -0.126  2.368 -0.663 -0.47  -0.17   0.371]
 [-1.449  1.854 -1.015  0.069  0.865 -0.151 -0.654  0.482]
 [ 0.03  -0.142 -0.557 -1.958  0.805  0.722  1.565 -0.465]
 [-1.128 -0.008 -0.206  1.507 -1.47   0.903 -0.623  1.025]
 [-0.514 -1.759  0.685 -0.583  0.8   -0.847  1.183  1.037]]
────────────────────────────────────────────────────────────
Input  : 'the dog ran fast'
Tokens : ['the', 'dog', 'ran', 'fast']
Output shape : (4, 8)
Encoder output:
[[-1.263 -0.083 -0.167  2.393 -0.481 -0.55  -0.147  0.298]
 [ 0.679  0.572  0.018  1.256 -2.068 -0.137 -0.963  0.643]
 [ 0.025 -1.187  0.285  1.569 -1.777  0.859 -0.137  0.364]
 [-0.617 -0.142  1.122  1.84   0.358 -0.279 -0.87  -1.412]]
────────────────────────────────────────────────────────────
Input  : 'the animal didn't cross the street'
Tokens : ['the', 'animal', "didn't", 'cross', 'the', 'street']
Output shape : (6, 

# Section 2 — Decoder

## What is the Decoder?

The Encoder reads and understands the input sentence.
The Decoder **generates the output** — one token at a time.

In a translation task for example:

```
Encoder input  →  "the cat sat on mat"  (English)
Decoder output →  "le chat était assis sur le tapis"  (French)
```

The Decoder generates each output token by looking at:
1. **What it has generated so far** — via Masked Multi-Head Attention
2. **The full encoder output** — via Cross Attention

---

## Decoder vs Encoder — Key Differences

| | Encoder | Decoder |
|---|---|---|
| Input | Source sentence | Target sentence (shifted right) |
| Attention 1 | Self attention (full) | Masked self attention |
| Attention 2 | — | Cross attention with encoder output |
| Output | Context vectors | Probability over vocab |

---

## Cell 12 — Output Embedding + Positional Encoding

### Outputs Shifted Right — What Does This Mean?

This is the most important concept to understand before the decoder.

During **training**, the decoder receives the target sentence shifted one position to the right:

```
Target sentence    :  "le chat était assis"
Decoder input      :  <START> "le chat était"      ← shifted right
Decoder output     :  "le chat était assis"         ← predicts next token
```

We add a `<START>` token at the beginning and shift everything right by one.
This way at each position the decoder predicts the **next token**
using only the tokens that came before it — never peeking at future tokens.

At **inference time** the decoder generates autoregressively:
- Step 1 → input `<START>` → predicts `"le"`
- Step 2 → input `<START> "le"` → predicts `"chat"`
- Step 3 → input `<START> "le chat"` → predicts `"était"`
- ... and so on until `<END>` token is generated

---

### Autoregressive vs Teacher Forcing

#### During Inference — Autoregressive
The decoder is **autoregressive** — it generates one token at a time,
feeding its own previous output back as input for the next step:

```
Step 1 → input <START>              → predicts "le"
Step 2 → input <START> "le"         → predicts "chat"
Step 3 → input <START> "le" "chat"  → predicts "était"
...until <END> is generated
```

Each step depends on the previous — it cannot be parallelized.

---

#### During Training — Teacher Forcing (Non Autoregressive)
During training the decoder is **not autoregressive**.
Instead of feeding its own predictions back, we feed the
**ground truth target tokens** directly as input — this is called **Teacher Forcing**.

```
Ground truth  :  "le chat était assis"
Decoder input :  <START> "le" "chat" "était"   ← always the real tokens, never predicted ones
Decoder output:  "le" "chat" "était" "assis"   ← predicts all positions in parallel
```

The entire target sequence is available at once so all positions
are predicted **simultaneously in one forward pass** — fully parallelized.

---

#### Why Not Autoregressive During Training?

| | Autoregressive | Teacher Forcing |
|---|---|---|
| Input to decoder | Own previous predictions | Ground truth tokens |
| Parallelizable? | No — sequential | Yes — all at once |
| Training speed | Very slow | Fast |
| Computational cost | Very high | Low |

If we used autoregressive decoding during training:
- Step 2 would wait for Step 1 to finish
- Step 3 would wait for Step 2 to finish
- For a sequence of length 50 that is **50 sequential steps per sentence**
- Across millions of training sentences this becomes **computationally infeasible**

Since we already have the full ground truth target during training,
autoreggressiveness adds **zero benefit** and only increases computational cost.
Teacher Forcing gives us the same supervision signal at a fraction of the cost.

> **Note:** The shift right ensures that even with teacher forcing,
> at position $i$ the decoder only sees tokens $0$ to $i-1$ —
> it never sees the token it is trying to predict.
> This is enforced by the **mask** in Masked Multi-Head Attention (next cell).

### Embedding + Positional Encoding

Identical to the encoder side — the shifted target tokens are:
1. Looked up in an embedding table
2. Scaled by $\sqrt{d_{model}}$
3. Added with positional encoding

$$X_{decoder} = \text{Embedding}(y_{shifted}) \times \sqrt{d_{model}} + PE$$

> The decoder has its **own** embedding table — separate from the encoder.
> In some implementations they are tied (shared weights) but kept separate here
> for clarity.

---

## Roadmap — Decoder
- [x] Output Embedding + Positional Encoding
- [ ] Masked Multi-Head Attention + Add & Norm
- [ ] Cross Attention + Add & Norm
- [ ] Feed Forward + Add & Norm
- [ ] Linear + Softmax
- [ ] Full Decoder

In [12]:
# ── Output Embedding + Positional Encoding ───────────────────────

class OutputEmbedding:
    def __init__(self, vocab, d_model):
        """
        Args:
            vocab   : list of target vocabulary words including <START> and <END>
            d_model : embedding dimension
        """
        self.vocab        = {word: idx for idx, word in enumerate(vocab)}
        self.vocab_size   = len(vocab)
        self.d_model      = d_model

        # separate embedding table from encoder
        self.embedding_table = np.random.randn(len(vocab), d_model)
        self.scale           = np.sqrt(d_model)
        self.pe              = PositionalEncoding(d_model=d_model, max_seq_len=100)

    def shift_right(self, tokens):
        """
        Shift target tokens right by one — prepend <START>.

        Args:
            tokens  : list of target token strings e.g. ["le", "chat", "était"]
        Returns:
            shifted : list with <START> prepended e.g. ["<START>", "le", "chat"]
        """
        return ["<START>"] + tokens[:-1]

    def tokenize(self, tokens):
        """
        Convert list of token strings → list of integer indices.

        Args:
            tokens : list of strings
        Returns:
            list of integer indices
        """
        for t in tokens:
            if t not in self.vocab:
                raise ValueError(f"Unknown token: '{t}'. Add it to vocab.")
        return [self.vocab[t] for t in tokens]

    def forward(self, sentence, training=True):
        """
        Full pipeline: target sentence → shifted → embed → + PE.

        Args:
            sentence : plain string of target tokens
            training : if True apply shift right, else use as is (inference)
        Returns:
            X        : (seq_len, d_model) — ready for decoder
            tokens   : list of token strings after shifting
        """
        tokens = sentence.lower().split()

        # shift right during training
        if training:
            tokens = self.shift_right(tokens)

        token_ids = self.tokenize(tokens)

        # lookup + scale
        X = self.embedding_table[token_ids] * self.scale   # (seq_len, d_model)

        # add positional encoding
        X = self.pe.forward(X)                             # (seq_len, d_model)

        return X, tokens


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

target_vocab = ["<START>", "<END>", "le", "chat", "était", "assis", "sur", "tapis",
                "le", "chien", "courait", "vite"]

# remove duplicates while preserving order
target_vocab = list(dict.fromkeys(target_vocab))

d_model         = 8
target_sentence = "le chat était assis sur tapis"

out_embed       = OutputEmbedding(vocab=target_vocab, d_model=d_model)
X_dec, tokens   = out_embed.forward(target_sentence, training=True)

print(f"Original tokens  : {target_sentence.split()}")
print(f"Shifted tokens   : {tokens}")
print(f"Output shape     : {X_dec.shape}")
print(f"\nX_decoder (embedding + PE):\n{np.round(X_dec, 3)}")
print(f"\nX_decoder is now ready for Masked Multi-Head Attention.")

Original tokens  : ['le', 'chat', 'était', 'assis', 'sur', 'tapis']
Shifted tokens   : ['<START>', 'le', 'chat', 'était', 'assis', 'sur']
Output shape     : (6, 8)

X_decoder (embedding + PE):
[[ 1.405  0.609  1.832  5.308 -0.662  0.338  4.467  3.171]
 [-2.023  1.429 -2.468 -3.     4.155  0.361  0.192 -3.03 ]
 [-0.63  -0.102 -3.057  2.043 -1.679  0.175 -1.7    6.239]
 [ 0.103 -3.982  2.622 -2.498  0.621 -4.543 -3.754  1.557]
 [ 1.332 -0.169  0.062  0.069 -4.142 -1.037 -1.299  3.99 ]
 [ 0.013 -4.703  1.396 -0.212 -1.865  2.729  2.921  3.634]]

X_decoder is now ready for Masked Multi-Head Attention.


## Cell 13 — Masked Multi-Head Attention + Add & Norm

### What Happens Here?

This is the first sublayer of the decoder block.
It is identical to the encoder's Multi-Head Attention — with one critical difference: a **causal mask**.
X_decoder  ──────────────────────────────┐
│                                     │  (residual — skip connection)
▼                                     │
Masked Multi-Head Attention              │
│                                     │
▼                                     │
MHA_output + X_decoder  ←───────────────┘
│
▼
Layer Norm
│
▼
X_norm1   →  goes into Cross Attention (Cell 14)
---

### Why Masking?

In the encoder, every token can attend to every other token freely — full attention.

In the decoder, at position $i$ the model must **only attend to positions 0 through $i$** — never future tokens. This preserves the autoregressive property: during generation, future tokens do not exist yet.

Without masking, the decoder would cheat during training — position 2 could directly look at position 5's answer before predicting it.

---

### How the Mask Works

We create an **upper triangular mask** and set future positions to $-\infty$ before softmax:

$$\text{score}(i, j) = \begin{cases} QK^T_{ij} / \sqrt{d_k} & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

After softmax, $e^{-\infty} = 0$ — future tokens get exactly zero attention weight.

For `"<START> le chat"` the mask looks like:

|  | **\<START\>** | **le** | **chat** |
|---|---|---|---|
| **\<START\>** | ✓ | ✗ | ✗ |
| **le** | ✓ | ✓ | ✗ |
| **chat** | ✓ | ✓ | ✓ |

Each token can only attend to itself and all **previous** tokens — never future ones.

---

### Add & Norm — Same as Encoder

$$X_{norm1} = \text{LayerNorm}(X_{decoder} + \text{MaskedMHA}(X_{decoder}))$$

Same residual connection and Layer Norm pattern as the encoder.

---

## Roadmap — Decoder
- [x] Output Embedding + Positional Encoding
- [x] Masked Multi-Head Attention + Add & Norm
- [ ] Cross Attention + Add & Norm
- [ ] Feed Forward + Add & Norm
- [ ] Linear + Softmax
- [ ] Full Decoder

In [13]:
# ── Masked Multi-Head Attention + Add & Norm ──────────────────────

class MaskedMultiHeadAttention:
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model   : total embedding dimension
            num_heads : number of parallel attention heads
        """
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.Wq = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wk = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wv = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wo = np.random.randn(d_model, d_model) * np.sqrt(2.0 / (d_model * 2))

    def _causal_mask(self, seq_len):
        """
        Upper triangular mask — blocks attention to future positions.

        Returns:
            mask : (seq_len, seq_len) — 0 for allowed, -inf for blocked
        """
        mask = np.zeros((seq_len, seq_len))
        mask[np.triu_indices(seq_len, k=1)] = -np.inf   # future positions → -inf
        return mask

    def _single_head_attention(self, X, Wq, Wk, Wv, mask):
        """
        Single-head attention with causal mask applied before softmax.

        Args:
            X        : (seq_len, d_model)
            Wq,Wk,Wv : (d_model, d_k)
            mask     : (seq_len, seq_len)
        Returns:
            output   : (seq_len, d_k)
            weights  : (seq_len, seq_len)
        """
        Q = X @ Wq
        K = X @ Wk
        V = X @ Wv

        scores = Q @ K.T / np.sqrt(self.d_k)   # (seq_len, seq_len)
        scores = scores + mask                  # add -inf to future positions

        # numerical stability — subtract max before exp
        scores = scores - np.where(np.isinf(scores), 0, scores).max(axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        exp_scores = np.where(np.isinf(scores), 0, exp_scores)  # keep -inf positions as 0
        weights = exp_scores / (exp_scores.sum(axis=1, keepdims=True) + 1e-9)

        output = weights @ V
        return output, weights

    def forward(self, X):
        """
        Run all masked heads in parallel, concat, then project.

        Args:
            X      : (seq_len, d_model)
        Returns:
            output : (seq_len, d_model)
            heads  : list of attention weight matrices per head
        """
        seq_len = X.shape[0]
        mask    = self._causal_mask(seq_len)   # (seq_len, seq_len)

        head_outputs = []
        head_weights = []

        for i in range(self.num_heads):
            out, weights = self._single_head_attention(X, self.Wq[i], self.Wk[i], self.Wv[i], mask)
            head_outputs.append(out)
            head_weights.append(weights)

        concat = np.concatenate(head_outputs, axis=-1)   # (seq_len, d_model)
        output = concat @ self.Wo                        # (seq_len, d_model)

        return output, head_weights


class MaskedMHAWithAddNorm:
    def __init__(self, d_model, num_heads):
        self.masked_mha = MaskedMultiHeadAttention(d_model, num_heads)
        self.norm       = LayerNorm(d_model)

    def forward(self, X):
        """
        Args:
            X       : (seq_len, d_model) — output of OutputEmbedding
        Returns:
            X_norm1 : (seq_len, d_model) — ready for Cross Attention
        """
        mha_out, attn_weights = self.masked_mha.forward(X)    # masked attention
        X_norm1 = self.norm.forward(X + mha_out)              # residual + norm
        return X_norm1, attn_weights


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

target_vocab    = ["<START>", "<END>", "le", "chat", "était", "assis", "sur", "tapis",
                   "chien", "courait", "vite"]
target_sentence = "le chat était assis sur tapis"
d_model         = 8
num_heads       = 2

out_embed             = OutputEmbedding(vocab=target_vocab, d_model=d_model)
X_dec, tokens         = out_embed.forward(target_sentence, training=True)

masked_mha_norm       = MaskedMHAWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm1, attn_weights = masked_mha_norm.forward(X_dec)

print(f"Shifted tokens   : {tokens}")
print(f"Input  shape     : {X_dec.shape}")
print(f"Output shape     : {X_norm1.shape}")
print(f"\nCausal mask (seq_len={len(tokens)}):")
print(masked_mha_norm.masked_mha._causal_mask(len(tokens)))
print(f"\nHead 1 attention weights (lower triangular = causal):\n{np.round(attn_weights[0], 3)}")
print(f"\nX_norm1 ready for Cross Attention:\n{np.round(X_norm1, 3)}")

Shifted tokens   : ['<START>', 'le', 'chat', 'était', 'assis', 'sur']
Input  shape     : (6, 8)
Output shape     : (6, 8)

Causal mask (seq_len=6):
[[  0. -inf -inf -inf -inf -inf]
 [  0.   0. -inf -inf -inf -inf]
 [  0.   0.   0. -inf -inf -inf]
 [  0.   0.   0.   0. -inf -inf]
 [  0.   0.   0.   0.   0. -inf]
 [  0.   0.   0.   0.   0.   0.]]

Head 1 attention weights (lower triangular = causal):
[[1.    0.    0.    0.    0.    0.   ]
 [0.968 0.032 0.    0.    0.    0.   ]
 [0.    0.002 0.998 0.    0.    0.   ]
 [1.    0.    0.    0.    0.    0.   ]
 [0.    0.001 0.996 0.001 0.002 0.   ]
 [1.    0.    0.    0.    0.    0.   ]]

X_norm1 ready for Cross Attention:
[[-0.486 -0.964 -0.441  1.927 -1.171  1.24  -0.132  0.027]
 [-0.792 -0.153 -0.925  0.626  0.736  2.071 -0.629 -0.935]
 [-0.593 -0.404 -1.139  1.287 -0.358  1.091 -1.209  1.323]
 [ 1.336 -1.759  0.666 -1.207 -0.247  0.808 -0.258  0.663]
 [ 0.88   0.288 -0.987 -0.635 -1.794  0.825  0.057  1.366]
 [-0.177 -1.421 -0.224 -0.116 -1

## Cell 14 — Cross Attention + Add & Norm

### What Happens Here?

This is the second sublayer of the decoder block.
The decoder now looks at the **encoder's output** — this is where the two sides of the Transformer meet.

```
X_norm1 ──────────────────────────────────────┐
    │                                         │ (residual)
    ▼                                         │
Cross Attention                               │
    Q  ←── X_norm1      (decoder questions)   │
    K  ←── encoder_out  (encoder context)     │
    V  ←── encoder_out  (encoder answers)     │
    │                                         │
    ▼                                         │
cross_out + X_norm1 ←─────────────────────────┘
    │
    ▼
Layer Norm
    │
    ▼
X_norm2  →  Feed Forward + Add & Norm (Cell 15)
```

---

### Five Differences — Multi-Head vs Masked Multi-Head vs Cross Attention

| | **Multi-Head Attention** | **Masked Multi-Head Attention** | **Cross Attention** |
|---|---|---|---|
| **Where used** | Encoder | Decoder (1st sublayer) | Decoder (2nd sublayer) |
| **Q comes from** | Encoder input | Decoder input | Decoder (X_norm1) |
| **K, V come from** | Encoder input | Decoder input | Encoder output |
| **Mask applied?** | No — attends to all tokens freely | Yes — causal mask blocks future tokens | No — attends to full encoder output |
| **Purpose** | Each token understands context within the source | Each decoder token only sees past generated tokens | Decoder queries the encoder — bridges source and target |

---

### Why Q from Decoder, K and V from Encoder?

This is the core idea of Cross Attention — the decoder **asks questions** using its own representation, and the encoder **provides answers** using its context vectors.

| Symbol | Source | Meaning |
|--------|--------|---------|
| $Q$ | Decoder (`X_norm1`) | *"What information do I need from the source?"* |
| $K$ | Encoder output | *"What information does each source token have?"* |
| $V$ | Encoder output | *"What should I actually read from each source token?"* |

For a translation task `"the cat sat"` → `"le chat était"`:

When the decoder is generating `"chat"`, its Query vector asks:
*"Which source token is most relevant to what I am generating right now?"*

The encoder's Keys answer: *"'cat' is the most relevant source token."*

The encoder's Values deliver: *"Here is the full context vector for 'cat'."*

---

### Add & Norm — Same Pattern

$$X_{norm2} = \text{LayerNorm}(X_{norm1} + \text{CrossAttention}(X_{norm1},\ \text{enc\_out}))$$

Residual comes from `X_norm1` — the decoder's own representation before cross attention.

---

## Roadmap — Decoder
- [x] Output Embedding + Positional Encoding
- [x] Masked Multi-Head Attention + Add & Norm
- [x] Cross Attention + Add & Norm
- [ ] Feed Forward + Add & Norm
- [ ] Linear + Softmax
- [ ] Full Decoder

In [14]:
# ── Cross Attention + Add & Norm ──────────────────────────────────

class CrossAttention:
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model   : total embedding dimension
            num_heads : number of parallel attention heads
        """
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        scale = np.sqrt(2.0 / (d_model + self.d_k))

        # Q comes from decoder, K and V from encoder — separate projections
        self.Wq = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wk = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wv = [np.random.randn(d_model, self.d_k) * scale for _ in range(num_heads)]
        self.Wo = np.random.randn(d_model, d_model) * np.sqrt(2.0 / (d_model * 2))

    def _single_head_cross_attention(self, X_dec, X_enc, Wq, Wk, Wv):
        """
        Single-head cross attention — Q from decoder, K/V from encoder.
        No mask — decoder attends to ALL encoder positions freely.

        Args:
            X_dec    : (tgt_seq_len, d_model) — decoder query source
            X_enc    : (src_seq_len, d_model) — encoder key/value source
            Wq,Wk,Wv : (d_model, d_k)
        Returns:
            output   : (tgt_seq_len, d_k)
            weights  : (tgt_seq_len, src_seq_len)
        """
        Q = X_dec @ Wq    # (tgt_seq_len, d_k) — decoder asks questions
        K = X_enc @ Wk    # (src_seq_len, d_k) — encoder advertises content
        V = X_enc @ Wv    # (src_seq_len, d_k) — encoder provides answers

        # scores shape: (tgt_seq_len, src_seq_len)
        # each decoder token scored against every encoder token
        scores = Q @ K.T / np.sqrt(self.d_k)

        # no mask — decoder can attend to full source sequence
        scores  = scores - scores.max(axis=1, keepdims=True)
        weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)

        output  = weights @ V     # (tgt_seq_len, d_k)
        return output, weights

    def forward(self, X_dec, X_enc):
        """
        Run all cross attention heads in parallel, concat, then project.

        Args:
            X_dec  : (tgt_seq_len, d_model) — from Masked MHA + Add & Norm
            X_enc  : (src_seq_len, d_model) — encoder output
        Returns:
            output : (tgt_seq_len, d_model)
            heads  : list of attention weight matrices per head
        """
        head_outputs = []
        head_weights = []

        for i in range(self.num_heads):
            out, weights = self._single_head_cross_attention(
                X_dec, X_enc, self.Wq[i], self.Wk[i], self.Wv[i]
            )
            head_outputs.append(out)
            head_weights.append(weights)

        concat = np.concatenate(head_outputs, axis=-1)   # (tgt_seq_len, d_model)
        output = concat @ self.Wo                        # (tgt_seq_len, d_model)

        return output, head_weights


class CrossAttentionWithAddNorm:
    def __init__(self, d_model, num_heads):
        self.cross_attn = CrossAttention(d_model, num_heads)
        self.norm       = LayerNorm(d_model)

    def forward(self, X_norm1, encoder_out):
        """
        Args:
            X_norm1     : (tgt_seq_len, d_model) — from Masked MHA + Add & Norm
            encoder_out : (src_seq_len, d_model) — full encoder output
        Returns:
            X_norm2     : (tgt_seq_len, d_model) — ready for Feed Forward
        """
        cross_out, attn_weights = self.cross_attn.forward(X_norm1, encoder_out)
        X_norm2 = self.norm.forward(X_norm1 + cross_out)    # residual + norm
        return X_norm2, attn_weights


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

# source side — encoder
src_vocab    = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast",
                "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"]
src_sentence = "the cat sat on mat"
d_model      = 8
num_heads    = 2

encoder      = Encoder(vocab=src_vocab, d_model=d_model, num_heads=num_heads, d_ff=32)
encoder_out, src_tokens = encoder.forward(src_sentence)

# target side — decoder so far
target_vocab    = ["<START>", "<END>", "le", "chat", "était", "assis", "sur",
                   "tapis", "chien", "courait", "vite"]
target_sentence = "le chat était assis sur tapis"

out_embed             = OutputEmbedding(vocab=target_vocab, d_model=d_model)
X_dec, tgt_tokens     = out_embed.forward(target_sentence, training=True)

masked_mha_norm       = MaskedMHAWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm1, _            = masked_mha_norm.forward(X_dec)

# cross attention
cross_norm            = CrossAttentionWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm2, cross_weights = cross_norm.forward(X_norm1, encoder_out)

print(f"Source tokens    : {src_tokens}")
print(f"Target tokens    : {tgt_tokens}")
print(f"\nEncoder out shape : {encoder_out.shape}   ← K, V source")
print(f"X_norm1 shape     : {X_norm1.shape}    ← Q source")
print(f"X_norm2 shape     : {X_norm2.shape}    ← output")
print(f"\nCross Attention weights — Head 1")
print(f"rows = decoder tokens, cols = encoder tokens")
print(f"shape: {cross_weights[0].shape}")
print(np.round(cross_weights[0], 3))
print(f"\nX_norm2 ready for Feed Forward:\n{np.round(X_norm2, 3)}")

Source tokens    : ['the', 'cat', 'sat', 'on', 'mat']
Target tokens    : ['<START>', 'le', 'chat', 'était', 'assis', 'sur']

Encoder out shape : (5, 8)   ← K, V source
X_norm1 shape     : (6, 8)    ← Q source
X_norm2 shape     : (6, 8)    ← output

Cross Attention weights — Head 1
rows = decoder tokens, cols = encoder tokens
shape: (6, 5)
[[0.01  0.241 0.658 0.014 0.077]
 [0.601 0.081 0.023 0.267 0.028]
 [0.381 0.276 0.094 0.182 0.068]
 [0.012 0.176 0.512 0.023 0.277]
 [0.131 0.196 0.212 0.118 0.343]
 [0.058 0.202 0.59  0.083 0.067]]

X_norm2 ready for Feed Forward:
[[ 0.449  0.265 -0.56   0.389  0.734  1.59  -1.226 -1.64 ]
 [ 0.081  0.515 -1.201 -0.008  1.065 -1.745 -0.151  1.443]
 [ 0.871  0.818 -1.03   0.33   0.759 -2.065 -0.346  0.665]
 [ 0.236 -0.594  1.294 -0.845 -0.158  1.735 -0.212 -1.457]
 [ 1.844 -0.996 -1.166 -0.186  0.226  1.076 -0.985  0.188]
 [-0.526  1.274 -0.344  0.713  1.612 -0.397 -1.185 -1.146]]


## Cell 15 — Feed Forward + Add & Norm

### What Happens Here?

Identical to the encoder's Feed Forward sublayer — but now inside the **decoder block**.
Takes `X_norm2` from Cross Attention and applies a two-layer transformation, then residual + norm.

```
X_norm2 (from Cross Attention) ────────────────┐
    │                                          │ (residual)
    ▼                                          │
Linear Layer 1  →  expand  (d_model → d_ff)    │
    │                                          │
    ▼                                          │
ReLU                                           │
    │                                          │
    ▼                                          │
Linear Layer 2  →  compress  (d_ff → d_model)  │
    │                                          │
    ▼                                          │
ffn_out + X_norm2 ←────────────────────────────┘
    │
    ▼
Layer Norm
    │
    ▼
X_norm3  →  Linear + Softmax (Cell 16)
```

---

### Formula

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

$$X_{norm3} = \text{LayerNorm}(X_{norm2} + \text{FFN}(X_{norm2}))$$

---

### Encoder FFN vs Decoder FFN — Are They Different?

No — the math is identical. The only difference is **what flows into them**.

| | Encoder FFN | Decoder FFN |
|---|---|---|
| Input | `X_norm1` — after self attention | `X_norm2` — after cross attention |
| Weights | Encoder's own $W_1, W_2$ | Decoder's own $W_1, W_2$ |
| Purpose | Transform source token representations | Transform target token representations enriched with source context |

Each has its **own learned weights** — they do not share parameters.

---

### Why FFN After Cross Attention?

After Cross Attention, each decoder token has absorbed relevant information
from the encoder. But Cross Attention is still a **linear operation** — it can only
compute weighted sums of encoder values.

The FFN adds **non-linearity** — letting the model learn complex transformations
on top of that blended representation before making the final prediction.

Think of it as:
- Cross Attention → *"gather what I need from the source"*
- FFN → *"process and transform what I just gathered"*

---

## Roadmap — Decoder
- [x] Output Embedding + Positional Encoding
- [x] Masked Multi-Head Attention + Add & Norm
- [x] Cross Attention + Add & Norm
- [x] Feed Forward + Add & Norm
- [ ] Linear + Softmax
- [ ] Full Decoder

In [15]:
# ── Feed Forward + Add & Norm (Decoder) ──────────────────────────

class DecoderFFNWithAddNorm:
    def __init__(self, d_model, d_ff):
        """
        Args:
            d_model : embedding dimension
            d_ff    : inner dimension — typically 4 × d_model
        """
        self.ffn  = FeedForward(d_model, d_ff)   # reuse FeedForward from Cell 9
        self.norm = LayerNorm(d_model)            # reuse LayerNorm from Cell 8

    def forward(self, X_norm2):
        """
        Args:
            X_norm2 : (tgt_seq_len, d_model) — from Cross Attention + Add & Norm
        Returns:
            X_norm3 : (tgt_seq_len, d_model) — ready for Linear + Softmax
        """
        # step 1 — feed forward (expand → ReLU → compress)
        ffn_out = self.ffn.forward(X_norm2)                  # (tgt_seq_len, d_model)

        # step 2 — residual connection + layer norm
        X_norm3 = self.norm.forward(X_norm2 + ffn_out)       # (tgt_seq_len, d_model)

        return X_norm3


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

# source side — encoder
src_vocab    = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast",
                "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"]
src_sentence = "the cat sat on mat"
d_model      = 8
num_heads    = 2
d_ff         = 32   # 4 × d_model

encoder          = Encoder(vocab=src_vocab, d_model=d_model, num_heads=num_heads, d_ff=d_ff)
encoder_out, src_tokens = encoder.forward(src_sentence)

# target side — decoder so far
target_vocab    = ["<START>", "<END>", "le", "chat", "était", "assis", "sur",
                   "tapis", "chien", "courait", "vite"]
target_sentence = "le chat était assis sur tapis"

out_embed              = OutputEmbedding(vocab=target_vocab, d_model=d_model)
X_dec, tgt_tokens      = out_embed.forward(target_sentence, training=True)

masked_mha_norm        = MaskedMHAWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm1, _             = masked_mha_norm.forward(X_dec)

cross_norm             = CrossAttentionWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm2, _             = cross_norm.forward(X_norm1, encoder_out)

# feed forward + add & norm
dec_ffn_norm           = DecoderFFNWithAddNorm(d_model=d_model, d_ff=d_ff)
X_norm3                = dec_ffn_norm.forward(X_norm2)

print(f"Source tokens    : {src_tokens}")
print(f"Target tokens    : {tgt_tokens}")
print(f"\nX_norm2 shape (input)  : {X_norm2.shape}")
print(f"X_norm3 shape (output) : {X_norm3.shape}")
print(f"\nBefore FFN (X_norm2):\n{np.round(X_norm2, 3)}")
print(f"\nAfter  FFN + Add & Norm (X_norm3):\n{np.round(X_norm3, 3)}")
print(f"\nX_norm3 is ready for Linear + Softmax.")

Source tokens    : ['the', 'cat', 'sat', 'on', 'mat']
Target tokens    : ['<START>', 'le', 'chat', 'était', 'assis', 'sur']

X_norm2 shape (input)  : (6, 8)
X_norm3 shape (output) : (6, 8)

Before FFN (X_norm2):
[[ 0.449  0.265 -0.56   0.389  0.734  1.59  -1.226 -1.64 ]
 [ 0.081  0.515 -1.201 -0.008  1.065 -1.745 -0.151  1.443]
 [ 0.871  0.818 -1.03   0.33   0.759 -2.065 -0.346  0.665]
 [ 0.236 -0.594  1.294 -0.845 -0.158  1.735 -0.212 -1.457]
 [ 1.844 -0.996 -1.166 -0.186  0.226  1.076 -0.985  0.188]
 [-0.526  1.274 -0.344  0.713  1.612 -0.397 -1.185 -1.146]]

After  FFN + Add & Norm (X_norm3):
[[ 1.291  0.067 -0.008  0.409 -0.089  1.226 -0.967 -1.928]
 [ 0.27   1.174 -1.589  0.327  0.643 -1.368 -0.589  1.132]
 [ 0.84   1.249 -1.209  0.287  0.587 -1.665 -0.775  0.686]
 [ 0.587 -0.545  1.323 -0.542 -0.625  1.394  0.129 -1.721]
 [ 2.239 -0.491 -0.831 -0.991  0.201  0.708 -0.72  -0.114]
 [-0.188  1.646 -0.515  1.031  0.888 -0.437 -1.247 -1.179]]

X_norm3 is ready for Linear + Softmax.


## Cell 16 — Linear + Softmax

### What Happens Here?

This is the **final step** of the decoder.
`X_norm3` holds one vector per target token — rich with source context and learned transformations.
Now we convert each vector into a **probability distribution over the entire vocabulary**.

```
X_norm3  (tgt_seq_len, d_model)
    │
    ▼
Linear Projection  (d_model → vocab_size)
    │
    ▼
Softmax  (across vocab dimension)
    │
    ▼
Probabilities  (tgt_seq_len, vocab_size)
    │
    ▼
argmax → predicted token at each position
```

---

### The Formula

$$\text{logits} = X_{norm3} \cdot W_{out} + b_{out}$$

$$\text{probs} = \text{softmax}(\text{logits})$$

$$\hat{y}_t = \arg\max(\text{probs}_t)$$

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $X_{norm3}$ | `(tgt_seq_len, d_model)` | Decoder output |
| $W_{out}$ | `(d_model, vocab_size)` | Learned projection to vocab |
| logits | `(tgt_seq_len, vocab_size)` | Raw unnormalized scores per token |
| probs | `(tgt_seq_len, vocab_size)` | Probability over vocab at each position |
| $\hat{y}_t$ | scalar | Predicted token index at position $t$ |

---

### What Does Each Row Mean?

The output is `(tgt_seq_len, vocab_size)` — one row per target position.

Each row is a probability distribution — **the model's belief about what the next token should be** at that position.

For `"<START> le chat était assis sur"` as decoder input:

```
Position 0  (<START>)  →  highest prob on "le"
Position 1  (le)       →  highest prob on "chat"
Position 2  (chat)     →  highest prob on "était"
Position 3  (était)    →  highest prob on "assis"
Position 4  (assis)    →  highest prob on "sur"
Position 5  (sur)      →  highest prob on "tapis"
```

During **training** — cross entropy loss is computed between these probs and the ground truth.
During **inference** — we take the argmax (or sample) to get the next token.

---

### Training vs Inference

| | Training | Inference |
|---|---|---|
| All positions predicted | Yes — in parallel (teacher forcing) | No — one token at a time |
| Loss computed | Yes — cross entropy over all positions | No |
| Next input | Ground truth (teacher forcing) | Model's own prediction |

---

## Roadmap — Decoder
- [x] Output Embedding + Positional Encoding
- [x] Masked Multi-Head Attention + Add & Norm
- [x] Cross Attention + Add & Norm
- [x] Feed Forward + Add & Norm
- [x] Linear + Softmax
- [ ] Full Decoder

In [16]:
# ── Linear + Softmax ─────────────────────────────────────────────

class LinearSoftmax:
    def __init__(self, d_model, vocab_size):
        """
        Args:
            d_model    : input dimension from decoder
            vocab_size : number of tokens in target vocabulary
        """
        self.vocab_size = vocab_size

        # xavier initialization
        scale    = np.sqrt(2.0 / (d_model + vocab_size))
        self.W   = np.random.randn(d_model, vocab_size) * scale
        self.b   = np.zeros(vocab_size)

    def softmax(self, x):
        """
        Row-wise softmax with numerical stability.

        Args:
            x      : (tgt_seq_len, vocab_size)
        Returns:
            probs  : (tgt_seq_len, vocab_size) — each row sums to 1
        """
        x = x - x.max(axis=1, keepdims=True)   # stability trick
        e = np.exp(x)
        return e / e.sum(axis=1, keepdims=True)

    def forward(self, X_norm3):
        """
        Project decoder output to vocab and convert to probabilities.

        Args:
            X_norm3 : (tgt_seq_len, d_model)
        Returns:
            probs   : (tgt_seq_len, vocab_size)
            logits  : (tgt_seq_len, vocab_size) — raw scores before softmax
        """
        logits = X_norm3 @ self.W + self.b     # (tgt_seq_len, vocab_size)
        probs  = self.softmax(logits)           # (tgt_seq_len, vocab_size)
        return probs, logits


# ── Test ──────────────────────────────────────────────────────────

np.random.seed(42)

# source side — encoder
src_vocab    = ["the", "cat", "sat", "on", "mat", "dog", "ran", "fast",
                "animal", "didn't", "cross", "street", "because", "it", "was", "too", "tired"]
src_sentence = "the cat sat on mat"
d_model      = 8
num_heads    = 2
d_ff         = 32

encoder              = Encoder(vocab=src_vocab, d_model=d_model, num_heads=num_heads, d_ff=d_ff)
encoder_out, src_tokens = encoder.forward(src_sentence)

# target side — decoder so far
target_vocab    = ["<START>", "<END>", "le", "chat", "était", "assis", "sur",
                   "tapis", "chien", "courait", "vite"]
target_sentence = "le chat était assis sur tapis"

out_embed              = OutputEmbedding(vocab=target_vocab, d_model=d_model)
X_dec, tgt_tokens      = out_embed.forward(target_sentence, training=True)

masked_mha_norm        = MaskedMHAWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm1, _             = masked_mha_norm.forward(X_dec)

cross_norm             = CrossAttentionWithAddNorm(d_model=d_model, num_heads=num_heads)
X_norm2, _             = cross_norm.forward(X_norm1, encoder_out)

dec_ffn_norm           = DecoderFFNWithAddNorm(d_model=d_model, d_ff=d_ff)
X_norm3                = dec_ffn_norm.forward(X_norm2)

# linear + softmax
vocab_size             = len(target_vocab)
linear_softmax         = LinearSoftmax(d_model=d_model, vocab_size=vocab_size)
probs, logits          = linear_softmax.forward(X_norm3)

# decode predictions
predicted_ids          = np.argmax(probs, axis=1)
predicted_tokens       = [target_vocab[i] for i in predicted_ids]

print(f"Source tokens      : {src_tokens}")
print(f"Decoder input      : {tgt_tokens}")
print(f"\nlogits shape       : {logits.shape}")
print(f"probs  shape       : {probs.shape}")
print(f"\nProbability distributions (each row sums to 1):")
print(np.round(probs, 3))
print(f"\nRow sums (should all be 1.0): {np.round(probs.sum(axis=1), 3)}")
print(f"\nPredicted token ids : {predicted_ids}")
print(f"Predicted tokens    : {predicted_tokens}")
print(f"\nDecoder input  : {tgt_tokens}")
print(f"Predicted next : {predicted_tokens}")
print(f"\n(Weights are random — predictions are untrained. After training,")
print(f"position 0 should predict 'le', position 1 'chat', and so on.)")

Source tokens      : ['the', 'cat', 'sat', 'on', 'mat']
Decoder input      : ['<START>', 'le', 'chat', 'était', 'assis', 'sur']

logits shape       : (6, 11)
probs  shape       : (6, 11)

Probability distributions (each row sums to 1):
[[0.266 0.03  0.085 0.023 0.107 0.189 0.106 0.015 0.024 0.095 0.06 ]
 [0.053 0.286 0.067 0.159 0.043 0.021 0.054 0.035 0.146 0.055 0.08 ]
 [0.071 0.19  0.086 0.158 0.061 0.034 0.067 0.035 0.123 0.077 0.098]
 [0.132 0.01  0.075 0.022 0.102 0.317 0.116 0.065 0.021 0.086 0.053]
 [0.067 0.023 0.169 0.028 0.087 0.103 0.247 0.01  0.003 0.051 0.211]
 [0.19  0.111 0.027 0.059 0.037 0.036 0.012 0.02  0.429 0.066 0.014]]

Row sums (should all be 1.0): [1. 1. 1. 1. 1. 1.]

Predicted token ids : [0 1 1 5 6 8]
Predicted tokens    : ['<START>', '<END>', '<END>', 'assis', 'sur', 'chien']

Decoder input  : ['<START>', 'le', 'chat', 'était', 'assis', 'sur']
Predicted next : ['<START>', '<END>', '<END>', 'assis', 'sur', 'chien']

(Weights are random — predictions are untr

Source tokens      : ['the', 'cat', 'sat', 'on', 'mat']
Decoder input      : ['<start>', 'le', 'chat', 'était', 'assis', 'sur']

logits shape       : (6, 11)
probs  shape       : (6, 11)

Probability distributions (each row sums to 1):
[[0.082 0.091 0.094 0.079 0.101 0.088 0.096 0.112 0.076 0.089 0.092]
 [0.094 0.078 0.088 0.103 0.085 0.097 0.091 0.079 0.108 0.086 0.091]
 ...

Row sums (should all be 1.0): [1. 1. 1. 1. 1. 1.]

Predicted token ids : [6 3 0 8 2 7]
Predicted tokens    : ['sur', 'chat', '<START>', 'tapis', 'le', 'assis']

Decoder input  : ['<start>', 'le', 'chat', 'était', 'assis', 'sur']
Predicted next : ['sur', 'chat', '<START>', 'tapis', 'le', 'assis']

(Weights are random — predictions are untrained. After training,
position 0 should predict 'le', position 1 'chat', and so on.)

---

### In the Original Paper — Nx Blocks

The original Transformer stacks the decoder block **6 times (Nx = 6)**.
Each block receives:
- Output of the **previous decoder block** as its input
- The **same encoder output** for cross attention — unchanged across all blocks

For our implementation we use **1 block** — adding more is just a loop.

---

### What the Decoder Output Represents

At each position $t$, the output is a probability distribution over the full vocabulary.
The model is saying:

> *"Given everything I have read from the source and everything I have generated so far, here is my belief about what the next token should be."*

During training this is compared to the ground truth via **cross entropy loss**.
During inference we take **argmax** (greedy) or **sample** (beam search / temperature).

---

## Roadmap — Decoder Complete ✓
- [x] Output Embedding + Positional Encoding
- [x] Masked Multi-Head Attention + Add & Norm
- [x] Cross Attention + Add & Norm
- [x] Feed Forward + Add & Norm
- [x] Linear + Softmax
- [x] Full Decoder ✓



In [ ]:
s